# 02 · Main results

Reads `results/` and `reports/figures/` written by `python run_pipeline.py`. Nothing is re-estimated here. The primary interval is the **period wild cluster bootstrap**: about 4 effective periods identify the estimate, so trader-clustered intervals are too narrow.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for cand in (start, *start.parents):
        if (cand / "run_pipeline.py").is_file() and (cand / "src").is_dir():
            return cand
    return start


ROOT = find_repo_root()
RES, FIG = ROOT / "results", ROOT / "reports" / "figures"


def load_json(name):
    p = RES / name
    if not p.exists():
        print(f"missing {p.name}: run `python run_pipeline.py` from the repo root")
        return None
    return json.loads(p.read_text())


def load_csv(name):
    p = RES / name
    if not p.exists():
        print(f"missing {p.name}: run `python run_pipeline.py` from the repo root")
        return None
    return pd.read_csv(p)


def show(name, width=800):
    p = FIG / name
    if p.exists():
        display(Image(filename=str(p), width=width))
    else:
        print(f"missing figure {name}")


manifest = load_json("manifest.json")
if manifest:
    last = manifest["runs"][manifest["latest_run"]]
    print("Latest run:", last["run_started"], "| package", last["package_version"], "| commit", last["git_commit"])


## Sample and descriptives

In [ ]:
main = load_json("main_results.json")
pd.Series(main["sample"]).to_frame("rows / traders") if main else None

In [ ]:
pd.Series(main["descriptives"]).to_frame("value") if main else None

## Main estimates (effect of one extra self-trade on weekly gain)

In [ ]:
rows = []
for m in main["models"]:
    rows.append({"model": m["model_name"], "estimate": m["ate"],
                 "CI trader": (round(m["ate_ci_lower"], 2), round(m["ate_ci_upper"], 2)),
                 "CI period wild": (None if m.get("ate_ci_lower_week_wild") is None else
                                    (round(m["ate_ci_lower_week_wild"], 2), round(m["ate_ci_upper_week_wild"], 2))),
                 "wild p": m.get("ate_pvalue_week_wild"), "two-way p": m.get("ate_pvalue_twoway"),
                 "effective periods": m.get("effective_time_clusters")})
pd.DataFrame(rows).round(3)

In [ ]:
show("fig4_main_estimates.png", 760)

## Where the identifying variation comes from

In [ ]:
print("Aggregates:", main["week_aggregates"])
print("Break:", main["regime_break"])

In [ ]:
wk = load_csv("week_decomposition.csv")
wk.sort_values("weight", ascending=False).head(10)[["period", "start_date", "weight", "slope",
    "slope_ci_lower", "slope_ci_upper", "self_trades", "raw_self_gain_per_self_trade"]].round(3)

In [ ]:
load_csv("week_blocks.csv").round(3)

In [ ]:
show("fig5_week_decomposition.png", 900)

In [ ]:
show("fig2_activity.png", 760)

## Heterogeneity (held-out traders)

In [ ]:
het = load_json("heterogeneity.json")
print("Calibration:", {k: round(v, 3) for k, v in het["blp_calibration_heldout"].items()})
print("Forest importance:", het["causal_forest"]["feature_importances_named"])

In [ ]:
pd.DataFrame(het["gates_heldout"]["groups"])[["group", "gate", "ci_lower", "ci_upper",
    "share_self_trades_late_block"]].round(3)

In [ ]:
pd.DataFrame(het["blp_moderators_heldout"]["terms"]).round(3)

In [ ]:
show("fig7_gates.png", 560); show("fig8_heterogeneity.png", 800); show("fig6_cate_distribution.png", 640)